**V1**

# **Análise Léxica (Tokenizador)**
*   Separa texto em tokens:


In [ ]:
import re

TOKENS = [
    ('NUMBER',   r'\d+'),
    ('STRING',   r'"[^"]*"'),
    ('ID',       r'[a-zA-Z_][a-zA-Z0-9_]*'),
    ('OP',       r'==|!=|>=|<=|>|<'),
    ('ARROW',    r'->'),
    ('LBRACE',   r'\{'),
    ('RBRACE',   r'\}'),
    ('SEMICOL',  r';'),
    ('EQUAL',    r'='),
    ('SKIP',     r'[ \t\n]+'),
]

def tokenize(code):
    regex = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in TOKENS)
    tokens = []

    for match in re.finditer(regex, code):
        kind = match.lastgroup
        value = match.group()

        if kind != 'SKIP':
            tokens.append((kind, value))

    return tokens

# **Parser (início)**


*   Parser recursivo descendente
*   Utiliza uma lista de tokens gerada pelo lexer e transforma isso em uma estrutura de objetos da linguagem, chamado AST (Abstract Syntax Tree).


In [ ]:
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def current(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def eat(self, expected_type=None):
        token = self.current()
        if expected_type and token[0] != expected_type:
            raise SyntaxError(f"Esperado {expected_type}, encontrado {token}")
        self.pos += 1
        return token

    def parse_program(self):
        declarations = []
        while self.current():
            declarations.append(self.parse_declaration())
        return declarations

    def parse_declaration(self):
        token = self.current()

        if token[1] == "scene":
            return self.parse_scene()
        elif token[1] == "character":
            # Implement parsing for character if needed
            raise NotImplementedError("Parsing de 'character' não implementado ainda.")
        elif token[1] == "event":
            # Implement parsing for event if needed
            raise NotImplementedError("Parsing de 'event' não implementado ainda.")
        else:
            raise SyntaxError(f"Declaração inválida: {token}")

    def parse_scene(self):
        self.eat()  # scene
        name = self.eat('ID')[1]
        self.eat('LBRACE')

        commands = []
        while self.current()[0] != 'RBRACE':
            commands.append(self.parse_command())

        self.eat('RBRACE')
        return Scene(name, commands)

    def parse_command(self):
        token = self.current()

        if token[1] == "say":
            self.eat()
            text = self.eat('STRING')[1]
            self.eat('SEMICOL')
            return Say(text)

        elif token[1] == "goto":
            self.eat()
            target = self.eat('ID')[1]
            self.eat('SEMICOL')
            return Goto(target)

        elif token[1] == "trigger":
            self.eat()
            event = self.eat('ID')[1]
            self.eat('SEMICOL')
            return Trigger(event)

        else:
            raise SyntaxError(f"Comando inválido: {token}")

# **Estrutura da AST**

*  Abstract Syntax Tree
*  Árvore Sintática Abstrata
*  É uma representação estruturada do código fonte.



In [3]:
class Scene:
    def __init__(self, name, commands):
        self.name = name
        self.commands = commands

class Character:
    def __init__(self, name, attributes):
        self.name = name
        self.attributes = attributes

class Event:
    def __init__(self, name, actions):
        self.name = name
        self.actions = actions

class Say:
    def __init__(self, text):
        self.text = text

class Choice:
    def __init__(self, options):
        self.options = options

class Goto:
    def __init__(self, target):
        self.target = target

class Trigger:
    def __init__(self, event):
        self.event = event

# **Interpretador simples**

In [ ]:
class Interpreter:
    def __init__(self, ast):
        self.ast = ast
        self.scenes = {node.name: node for node in ast if isinstance(node, Scene)}

    def run(self, start_scene):
        current = self.scenes[start_scene]

        while True:
            for cmd in current.commands:
                if isinstance(cmd, Say):
                    print(cmd.text.strip('"'))

                elif isinstance(cmd, Goto):
                    current = self.scenes[cmd.target]
                    break
            else:
                break

# **Teste**

In [ ]:
code = '''
scene inicio {
    say "Bem-vindo ao jogo!";
    goto fim;
}

scene fim {
    say "Fim da aventura!";
}
'''

tokens = tokenize(code)
parser = Parser(tokens)
ast = parser.parse_program()

interpreter = Interpreter(ast)
interpreter.run("inicio")